In [ ]:
# ==========================================================
# HOUSE PRICE PREDICTION PROJECT
# ==========================================================

# ==========================
# 1. Import Libraries
# ==========================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_val_score

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score

# ==========================================================
# TASK 1 - DATA LOADING & EXPLORATION
# ==========================================================

print("="*60)
print("LOADING DATASET")
print("="*60)

df = pd.read_csv("housing.csv")

print("\nFirst 10 Rows:")
display(df.head(10))

print("\nDataset Shape:")
print(df.shape)

print("\nColumn Names:")
print(df.columns)

print("\nDataset Information:")
print(df.info())

print("\nStatistical Summary:")
display(df.describe())

print("\nMissing Values:")
print(df.isnull().sum())

print("\nOcean Proximity Counts:")
print(df["ocean_proximity"].value_counts())

# Target and Features

target = "median_house_value"

print("\nTarget Column:")
print(target)

print("\nFeature Columns:")
print(df.drop(target, axis=1).columns)

# ==========================================================
# TASK 2 - DATA CLEANING
# ==========================================================

print("\n" + "="*60)
print("DATA CLEANING")
print("="*60)

data = df.copy()

# Handle Missing Values

data["total_bedrooms"] = data["total_bedrooms"].fillna(
    data["total_bedrooms"].median()
)

# Remove Duplicates

data = data.drop_duplicates()

# Convert Categorical Data

data = pd.get_dummies(
    data,
    columns=["ocean_proximity"],
    drop_first=True
)

print("\nDataset Shape After Cleaning:")
print(data.shape)

print("\nRemaining Missing Values:")
print(data.isnull().sum().sum())

# ==========================================================
# CORRELATION ANALYSIS
# ==========================================================

print("\nTop Features Affecting House Price:")

correlation = data.corr()["median_house_value"] \
                  .sort_values(ascending=False)

print(correlation.head(10))

# ==========================================================
# TASK 3 - MODEL BUILDING
# ==========================================================

print("\n" + "="*60)
print("MODEL BUILDING")
print("="*60)

X = data.drop("median_house_value", axis=1)
y = data["median_house_value"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("Training Data Shape:", X_train.shape)
print("Testing Data Shape :", X_test.shape)

# ==========================================================
# LINEAR REGRESSION
# ==========================================================

lr = LinearRegression()

lr.fit(X_train, y_train)

lr_pred = lr.predict(X_test)

lr_mae = mean_absolute_error(y_test, lr_pred)
lr_rmse = np.sqrt(mean_squared_error(y_test, lr_pred))
lr_r2 = r2_score(y_test, lr_pred)

print("\nLINEAR REGRESSION RESULTS")
print("-"*40)

print("MAE :", lr_mae)
print("RMSE:", lr_rmse)
print("R2  :", lr_r2)

# ==========================================================
# RANDOM FOREST REGRESSOR
# ==========================================================

rf = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

rf.fit(X_train, y_train)

rf_pred = rf.predict(X_test)

rf_mae = mean_absolute_error(y_test, rf_pred)
rf_rmse = np.sqrt(mean_squared_error(y_test, rf_pred))
rf_r2 = r2_score(y_test, rf_pred)

print("\nRANDOM FOREST RESULTS")
print("-"*40)

print("MAE :", rf_mae)
print("RMSE:", rf_rmse)
print("R2  :", rf_r2)

# ==========================================================
# CROSS VALIDATION
# ==========================================================

scores = cross_val_score(
    rf,
    X,
    y,
    cv=5,
    scoring="r2"
)

print("\nCross Validation R2 Scores:")
print(scores)

print("\nAverage Cross Validation Score:")
print(scores.mean())

# ==========================================================
# MODEL COMPARISON
# ==========================================================

comparison = pd.DataFrame({
    "Model":["Linear Regression","Random Forest"],
    "MAE":[lr_mae,rf_mae],
    "RMSE":[lr_rmse,rf_rmse],
    "R2 Score":[lr_r2,rf_r2]
})

print("\nMODEL COMPARISON")
display(comparison)

# ==========================================================
# TASK 4 - VISUALIZATIONS
# ==========================================================

sns.set_style("whitegrid")

# -----------------------------
# Chart 1 - Price Distribution
# -----------------------------

plt.figure(figsize=(8,5))

sns.histplot(
    data["median_house_value"],
    bins=30,
    kde=True
)

plt.title("Distribution of House Prices")
plt.xlabel("House Price")
plt.ylabel("Count")

plt.savefig("price_distribution.png")

plt.show()

# -----------------------------
# Chart 2 - Correlation Heatmap
# -----------------------------

plt.figure(figsize=(12,8))

sns.heatmap(
    data.corr(),
    cmap="coolwarm"
)

plt.title("Correlation Heatmap")

plt.savefig("correlation_heatmap.png")

plt.show()

# -----------------------------
# Chart 3 - Actual vs Predicted
# -----------------------------

plt.figure(figsize=(8,5))

plt.scatter(
    y_test,
    rf_pred
)

plt.xlabel("Actual Prices")
plt.ylabel("Predicted Prices")

plt.title("Actual vs Predicted Prices")

plt.savefig("actual_vs_predicted.png")

plt.show()

# -----------------------------
# Chart 4 - Feature Importance
# -----------------------------

importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": rf.feature_importances_
})

importance = importance.sort_values(
    by="Importance",
    ascending=False
)

plt.figure(figsize=(10,6))

sns.barplot(
    data=importance.head(10),
    x="Importance",
    y="Feature"
)

plt.title("Top 10 Important Features")

plt.savefig("feature_importance.png")

plt.show()

# -----------------------------
# Chart 5 - Income vs Price
# -----------------------------

plt.figure(figsize=(8,5))

sns.scatterplot(
    x=data["median_income"],
    y=data["median_house_value"]
)

plt.title("Median Income vs House Price")

plt.savefig("income_vs_price.png")

plt.show()

# -----------------------------
# Chart 6 - Price Outliers
# -----------------------------

plt.figure(figsize=(8,5))

sns.boxplot(
    x=data["median_house_value"]
)

plt.title("House Price Outliers")

plt.savefig("price_outliers.png")

plt.show()

# ==========================================================
# TASK 5 - INSIGHTS & SUMMARY
# ==========================================================

print("\n" + "="*60)
print("PROJECT SUMMARY")
print("="*60)

print("""
1. The most influential features affecting house price were
   median income, location-related variables, total rooms,
   and housing age.

2. Random Forest Regressor performed better than
   Linear Regression with higher R² score and lower errors.

3. The strongest relationship with house price was
   median income.

4. House prices were highly dependent on both
   economic and geographical factors.

5. Real estate businesses should focus on income levels,
   location, and housing characteristics while estimating
   property prices.
""")

LOADING DATASET


FileNotFoundError: [Errno 2] No such file or directory: 'housing.csv'